# 项目 #1：AI IT Support Assistant

## 项目知识点

```text
AI IT Support Assistant
│
├── 1. OpenAI API
│
├── 2. System Instructions
│
├── 3. Prompt Engineering
│
├── 4. Structured Output
│
├── 5. JSON Schema
│
├── 6. Pydantic
│
├── 7. Input Validation
│
├── 8. Error Handling
│
├── 9. Python
│
└── 10. LLM Application Design
```

## 项目目标
用户输入：
```text
My laptop cannot connect to Wi-Fi.
```

AI 输出严格的 JSON：
```json
{
  "category": "network",
  "priority": "medium",
  "possible_causes": [
    "Wi-Fi adapter disabled",
    "Incorrect network configuration"
  ],
  "recommended_actions": [
    "Restart Wi-Fi adapter",
    "Check IP configuration"
  ]
}
```

## 项目设计

```text
User
  │
  │ IT problem
  ▼
Python Application
  │
  │ Prompt + Structured Output
  ▼
OpenAI API
  │
  │ JSON
  ▼
Python
  │
  ├── category
  ├── priority
  ├── possible_causes
  └── recommended_actions
```

In [3]:
# settings.py

from dotenv import load_dotenv

load_dotenv()

# 学习 / 写简单代码：
OPENAI_MODEL1 = "gpt-5.4-mini"

## 定义 AI 的任务

In [5]:
instructions = """
# Role
You are an AI IT support assistant.

# Objective
Your goal is to help customers solve their IT problems quickly, safely, and clearly.

# Responsibilities
You should:
- Understand the customer's IT problem.
- Ask clear questions when important information is missing.
- Provide step-by-step troubleshooting instructions.
- Explain technical concepts using simple language.
- Help customers identify the likely cause of an IT problem.
- Recommend the safest practical solution.

# Troubleshooting Process
When helping a customer:
1. Identify the problem.
2. Ask for missing information if necessary.
3. Start with simple and low-risk troubleshooting steps.
4. Explain each step clearly.
5. Check the result after each important step.
6. If the problem is not solved, continue with the next troubleshooting step.
7. If you cannot solve the problem, explain what has been checked and recommend contacting an IT technician.

# Rules
- Always be polite, professional, and patient.
- Do not guess when you do not have enough information.
- Do not provide unnecessary technical details unless the customer asks for them.
- Do not ask customers to perform dangerous or irreversible actions without explaining the risks.
- Never ask customers for passwords, authentication codes, private keys, or other sensitive credentials.

# Constraints
- Do not expose confidential or sensitive information.
- Do not make changes to systems, accounts, or security settings without the customer's confirmation and appropriate authorization.
- Do not claim that an issue has been fixed unless there is evidence that it has been resolved.
- If you are unsure about a solution, clearly state the uncertainty and recommend an appropriate next step.
"""


## 定义 JSON Schema
AI 不能随便返回文字，而必须返回
```json
{
    "category": "...",
    "priority": "...",
    "possible_causes": [],
    "recommended_actions": []
}
```

In [6]:
response_format = {
    "type": "json_schema",
    "name": "it_support_response",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "enum": [
                    "network",
                    "hardware",
                    "software",
                    "account",
                    "security",
                    "printer",
                    "other",
                ],
            },
            "priority": {
                "type": "string",
                "enum": [
                    "low",
                    "medium",
                    "high",
                    "critical",
                ],
            },
            "possible_causes": {
                "type": "array",
                "items": {
                    "type": "string",
                },
            },
            "recommended_actions": {
                "type": "array",
                "items": {
                    "type": "string",
                },
            },
        },
        "required": [
            "category",
            "priority",
            "possible_causes",
            "recommended_actions",
        ],
        "additionalProperties": False,
    },
}

## 调用 OpenAI API

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model=OPENAI_MODEL1,
    instructions=instructions,
    input="My Wi-Fi is not working.",
    text={"format": response_format},  # type: ignore
)

print(response.output_text)

{"category":"network","priority":"medium","possible_causes":["Wi-Fi router or modem is offline","Device Wi-Fi is turned off or in airplane mode","Weak signal or temporary interference","Incorrect network password or saved network issue","ISP outage or service disruption"],"recommended_actions":["Check whether Wi-Fi is turned on and airplane mode is off on your device","Look at the router/modem lights to see if they indicate a connection problem","Restart your device and then restart the router/modem by unplugging it for 30 seconds and plugging it back in","Try connecting another device to see if the issue is only on one device or on the whole network","Move closer to the router to rule out weak signal","If the network appears down on multiple devices, contact your internet service provider to check for an outage"]}


In [ ]:
## 让用户自己输入问题
user_input = input("Describe your IT problem: ")


def get_response():
    response = client.responses.create(
        model=OPENAI_MODEL1,
        instructions=instructions,
        input=user_input,  # My Wi-Fi is not working.
        text={"format": response_format},  # type: ignore
    )
    return response

In [9]:
response = get_response()
print(response.output_text)

{"category":"network","priority":"medium","possible_causes":["Wi-Fi router or access point is down","Device Wi-Fi adapter is turned off or malfunctioning","Incorrect Wi-Fi password or connection settings","Temporary network outage from the internet provider","Weak signal or interference from other devices"],"recommended_actions":["Check whether other devices can connect to the same Wi-Fi network","Restart your device and the Wi-Fi router/access point","Make sure Wi-Fi is turned on and airplane mode is off","Forget the Wi-Fi network and reconnect using the correct password","Move closer to the router to rule out weak signal or interference","If possible, test with another network to see whether the issue is with your device or the Wi-Fi itself","If none of these steps work, contact your internet provider or IT support to check for outages or router issues"]}


## 加入 Python JSON 处理 

In [12]:
import json

result = json.loads(response.output_text)

print(result["category"])
print(result["priority"])

print("\nPossible causes:")

for cause in result["possible_causes"]:
    print(f" - {cause}")

print("\nRecommended actions:")

for action in result["recommended_actions"]:
    print(f" - {action}")

network
medium

Possible causes:
 - Wi-Fi router or access point is down
 - Device Wi-Fi adapter is turned off or malfunctioning
 - Incorrect Wi-Fi password or connection settings
 - Temporary network outage from the internet provider
 - Weak signal or interference from other devices

Recommended actions:
 - Check whether other devices can connect to the same Wi-Fi network
 - Restart your device and the Wi-Fi router/access point
 - Make sure Wi-Fi is turned on and airplane mode is off
 - Forget the Wi-Fi network and reconnect using the correct password
 - Move closer to the router to rule out weak signal or interference
 - If possible, test with another network to see whether the issue is with your device or the Wi-Fi itself
 - If none of these steps work, contact your internet provider or IT support to check for outages or router issues


## 使用 Pydantic

数据结构：
```text
ITSupportResponse
│
├── category
├── priority
├── possible_causes
└── recommended_actions
```

In [13]:
from typing import Literal

from pydantic import BaseModel


class ITSupportResponse(BaseModel):
    category: Literal[
        "network",
        "hardware",
        "software",
        "account",
        "security",
        "printer",
        "other",
    ]

    priority: Literal[
        "low",
        "medium",
        "high",
        "critical",
    ]

    possible_casues: list[str]

    recommended_actions: list[str]

## 完整代码

In [2]:
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel

from settings import OPENAI_MODEL1

client = OpenAI()


class ITSupportResponse(BaseModel):
    category: Literal[
        "network",
        "hardware",
        "software",
        "account",
        "security",
        "printer",
        "other",
    ]

    priority: Literal[
        "low",
        "medium",
        "high",
        "critical",
    ]

    possible_causes: list[str]

    recommended_actions: list[str]


instructions = """
# Role
You are an AI IT support assistant.

# Objective
Analyze the user's IT problem and provide a structured troubleshooting response.

# Responsibilities
You should:
- Identify the category of the IT problem.
- Estimate the priority.
- Identify possible causes.
- Recommend practical troubleshooting actions.

# Categories
Use one of these categories:
- network
- hardware
- software
- account
- security
- printer
- other

# Priority
Use one of these priorities:
- low
- medium
- high
- critical

# Rules
- Keep the response concise.
- Do not invent information.
- Only recommend safe and common IT troubleshooting steps.
- Provide practical troubleshooting actions.
"""

user_input = input("Describe your IT problem: ")

response = client.responses.parse(
    model=OPENAI_MODEL1,
    instructions=instructions,
    input=user_input,  # My laptop cannot connect to Wi-Fi.
    text_format=ITSupportResponse,
)


result = response.output_parsed

if result is not None:
    print(result.model_dump_json(indent=4))

    print(f"\nCategory: {result.category}")
    print(f"Priority: {result.priority}")

    print("\nPossible causes:")

    for cause in result.possible_causes:
        print(f"- {cause}")

    print("\nRecommended actions:")

    for action in result.recommended_actions:
        print(f"- {action}")


{
    "category": "network",
    "priority": "medium",
    "possible_causes": [
        "Wi‑Fi adapter is turned off or disabled",
        "Incorrect Wi‑Fi password or saved network profile issue",
        "Router or access point is down",
        "Interference or weak signal",
        "IP address or network configuration problem",
        "Outdated or corrupted Wi‑Fi driver"
    ],
    "recommended_actions": [
        "Confirm Wi‑Fi is enabled on the laptop and airplane mode is off",
        "Restart the laptop and the router/access point",
        "Forget the Wi‑Fi network and reconnect with the correct password",
        "Move closer to the router to rule out signal issues",
        "Check whether other devices can connect to the same Wi‑Fi",
        "Run the built-in network troubleshooter",
        "If available, update or reinstall the Wi‑Fi adapter driver",
        "If the issue continues, try connecting to another network to isolate the problem"
    ]
}

Category: network
Prior